# Spatial uniformity around PhenoCam sites (geemap / VS Code)

Python + `geemap` port of `calcu_uniform.js`. It renders an **interactive Earth Engine map inline in VS Code** and prints per site NDVI uniformity (CV).

## One time setup
1. Install deps (from the repo root):
   ```
   pip install -r requirements.txt
   ```
2. Need a (free) Earth Engine account tied to a Google Cloud project:
   - Sign up / accept terms once at https://code.earthengine.google.com
   - Cloud project id (and make and .env file with 'EE_PROJECT = ...' field).
3. Run the **Authenticate & init** cell below once; a browser window opens for login.

Then run the cells top to bottom. Edit `SITE_NAMES` / `YEAR_FILTER` in the config cell.

In [ ]:
import os
import ee
import folium 
from dotenv import find_dotenv, load_dotenv

# Google Cloud project id
# .env file: EE_PROJECT= "your-project-id"
load_dotenv(find_dotenv())
EE_PROJECT = os.environ["EE_PROJECT"]

# auth_mode="localhost" avoids the "incompatible OAuth2 Client configuration" error
# opens a browser tab, to log in
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="localhost")
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized:", ee.String("ok").getInfo()) 

Earth Engine initialized: ok


In [7]:
import json
from pathlib import Path

# ---------
# CONFIG
# ---------
SITE_NAMES = ["goodwater_AG_1000"]
YEAR_FILTER = "2023"  # "2023", "2024", or "ALL"

# Read the sites produced by prep/api/main.py (grouped {group: {"sites": [...]}}).
SITE_JSON = Path("../../prep/output/api/site_GEE_clean.json")

data = json.loads(SITE_JSON.read_text())
all_sites = [site for group in data.values() for site in group["sites"]]


def year_matches(site):
    return YEAR_FILTER == "ALL" or str(site["metadata"]["year"]) == str(YEAR_FILTER)


matched_sites = [
    s for s in all_sites if s["name"] in SITE_NAMES and year_matches(s)
]

print(f"Loaded {len(all_sites)} sites, matched {len(matched_sites)}:")
for s in matched_sites:
    print(f"  {s['name']} ({s['metadata']['year']})  lat={s['lat']} lon={s['lon']}")

Loaded 40 sites, matched 1:
  goodwater_AG_1000 (2023)  lat=39.22848 lon=-92.11936


In [8]:
# ---------
# SPATIAL UNIFORMITY FUNCTION
# ---------
def uniformity(image, geom, scale, band_name):
    """Coefficient of variation (stdDev / mean) of a band over a region."""
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean().combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True),
        geometry=geom,
        scale=scale,
        maxPixels=int(1e9),
    )
    mean = ee.Number(stats.get(band_name + "_mean"))
    std = ee.Number(stats.get(band_name + "_stdDev"))
    return std.divide(mean)

In [ ]:
# ---------
# FOLIUM MAP WRAPPER
# ---------
# Minimal folium backed for the geemap Map API. Using folium (instead of
# geemap/ipyleaflet), uh something about the layers not rendering 
class FoliumMap:
    """Minimal folium wrapper exposing the geemap-style API used below."""

    def __init__(self, center, zoom):
        self.m = folium.Map(
            location=center,
            zoom_start=zoom,
            tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
            attr="Google Satellite",
        )

    def addLayer(self, ee_object, vis_params=None, name="Layer", shown=True):
        vis_params = dict(vis_params or {})
        if isinstance(ee_object, (ee.Geometry, ee.Feature, ee.FeatureCollection)):
            fc = ee.FeatureCollection(ee_object)
            color = vis_params.get("color") or (vis_params.get("palette") or ["FF0000"])[0]
            ee_object, vis_params = ee.Image().paint(fc, 0, 2), {"palette": [color]}
        mapid = ee.Image(ee_object).getMapId(vis_params)
        folium.raster_layers.TileLayer(
            tiles=mapid["tile_fetcher"].url_format,
            attr="Google Earth Engine",
            name=name,
            overlay=True,
            control=True,
            show=shown,
        ).add_to(self.m)

    def setCenter(self, lon, lat, zoom=None):
        self.m.location = [lat, lon]

    def add_layer_control(self):
        folium.LayerControl(collapsed=False).add_to(self.m)

    def to_html(self, path):
        self.m.save(path)

    def _repr_html_(self):
        return self.m._repr_html_()

In [ ]:
# ---------
# PER SITE LAYER GENERATION & UNIFORMITY MATH
# ---------
_center_lat = sum(s["lat"] for s in matched_sites) / len(matched_sites)
_center_lon = sum(s["lon"] for s in matched_sites) / len(matched_sites)
Map = FoliumMap([_center_lat, _center_lon], 13 if len(matched_sites) == 1 else 4)

all_footprint_points = []

for site in matched_sites:
    label = f"{site['name']} ({site['metadata']['year']})"
    lat, lon = site["lat"], site["lon"]
    year = str(site["metadata"]["year"])

    point = ee.Geometry.Point([lon, lat])
    camera_fov = point.buffer(200)
    satellite_pixel = point.buffer(2000).bounds()
    aoi = satellite_pixel

    start, end = f"{year}-01-01", f"{year}-12-31"
    all_footprint_points.append(point)

    s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 5))
        .median()
    )

    # 1: True Color
    Map.addLayer(s2.select(["B4", "B3", "B2"]).clip(aoi), {"min": 0, "max": 2000}, f"1 {label} - True Color", False)

    # 2: NDVI
    ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")
    Map.addLayer(
        ndvi.clip(aoi),
        {"min": -0.2, "max": 0.9, "palette": ["blue", "white", "yellow", "green", "darkgreen"]},
        f"2 {label} - NDVI", False,
    )

    # 3: GVF
    ndvi_bare, ndvi_veg = 0.05, 0.90
    gvf = ndvi.subtract(ndvi_bare).divide(ndvi_veg - ndvi_bare).pow(2).clamp(0, 1).rename("GVF")
    Map.addLayer(
        gvf.clip(aoi),
        {"min": 0, "max": 1, "palette": ["red", "orange", "yellow", "lightgreen", "darkgreen"]},
        f"3 {label} - GVF", False,
    )

    # 4: Spatial CV
    s2_collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 5))
        .map(lambda img: img.normalizedDifference(["B8", "B4"]).rename("NDVI"))
    )
    ndvi_mean = s2_collection.mean()
    ndvi_std = s2_collection.reduce(ee.Reducer.stdDev())
    spatial_cv = ndvi_std.divide(ndvi_mean).rename("CV")
    Map.addLayer(
        spatial_cv.clip(aoi),
        {"min": 0, "max": 0.5, "palette": ["darkgreen", "yellow", "orange", "red"]},
        f"4 {label} - Spatial CV", False,
    )

    # 5: Surface type flags
    mndwi = s2.normalizedDifference(["B3", "B11"]).rename("MNDWI")
    water_mask = mndwi.gt(0.0)
    bsi = s2.expression(
        "((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))",
        {"SWIR": s2.select("B11"), "RED": s2.select("B4"), "NIR": s2.select("B8"), "BLUE": s2.select("B2")},
    ).rename("BSI")
    bare_mask = bsi.gt(0.0)
    ndbi = s2.normalizedDifference(["B11", "B8"]).rename("NDBI")
    urban_mask = ndbi.gt(0.0)
    Map.addLayer(water_mask.clip(aoi).selfMask(), {"palette": ["0000FF"]}, f"5a {label} - WATER", False)
    Map.addLayer(bare_mask.clip(aoi).selfMask(), {"palette": ["964B00"]}, f"5b {label} - BARE SOIL", False)
    Map.addLayer(urban_mask.clip(aoi).selfMask(), {"palette": ["808080"]}, f"5c {label} - URBAN", False)

    # 6: Cloud frequency
    s2_all = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .filterDate(start, end)
    )
    cloud_freq = s2_all.map(lambda img: img.select("SCL").eq(9).rename("cloud")).mean()
    Map.addLayer(
        cloud_freq.clip(aoi),
        {"min": 0, "max": 0.5, "palette": ["white", "lightblue", "blue", "darkblue"]},
        f"6 {label} - Cloud frequency", False,
    )

    # 7: UNIFORMITY PRINTOUT (.getInfo() pulls the number back to Python)
    cv_satellite = uniformity(ndvi, satellite_pixel, 10, "NDVI").getInfo()
    cv_camera = uniformity(ndvi, camera_fov, 10, "NDVI").getInfo()
    print(f"{label} - satellite 4km uniformity (CV): {cv_satellite}")
    print(f"{label} - camera 200m uniformity (CV):  {cv_camera}")

    # 8: Footprints
    Map.addLayer(
        ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(satellite_pixel)]), 1, 2),
        {"palette": ["FFA500"]}, f"{label} - VIIRS pixel", True,
    )
    Map.addLayer(
        ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(camera_fov)]), 1, 2),
        {"palette": ["FF0000"]}, f"{label} - PhenoCam FOV", True,
    )
    Map.addLayer(point, {"color": "white"}, f"{label} - Camera", True)

# ---------
# MAP VIEW
# ---------
if matched_sites:
    center_lat = sum(s["lat"] for s in matched_sites) / len(matched_sites)
    center_lon = sum(s["lon"] for s in matched_sites) / len(matched_sites)
    Map.setCenter(center_lon, center_lat, 13 if len(matched_sites) == 1 else 4)

# Folium toggle layers on/off
Map.add_layer_control()

goodwater_AG_1000 (2023) - satellite 4km uniformity (CV): 0.3281056082018945
goodwater_AG_1000 (2023) - camera 200m uniformity (CV):  0.1364052152443128


In [17]:
# render it to an HTML file and open it in browser
import os
import webbrowser

out_html = os.path.abspath("uniformity_map.html")
Map.to_html(out_html)
print("Wrote", out_html)
webbrowser.open("file://" + out_html)

Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/test/GEE/uniformity_map.html


True